# S10 — AndinaLog 03B | Regularización y evaluación final de la MLP

Modelo y umbral se seleccionan con VALIDACIÓN. TEST se abre una sola vez después de congelar ambas decisiones.

## 1. Entorno y contrato

Se mantiene el objetivo de desviación térmica en 60 minutos, el split temporal por viaje y un costo didáctico de 10 por falso negativo y 1 por falso positivo.

In [1]:
from pathlib import Path
import os, sys, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
os.environ["TF_DETERMINISTIC_OPS"]="1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

ENTORNO="auto"; RUTA_PROYECTO_DRIVE="/content/drive/MyDrive/GIAD"
SEMILLA=42; TARGET="clasificacion_objetivo_60min"; COSTO_FN=10; COSTO_FP=1
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

def encontrar_raiz():
    if ENTORNO=="drive" or (ENTORNO=="auto" and "google.colab" in sys.modules):
        from google.colab import drive; drive.mount("/content/drive"); raiz=Path(RUTA_PROYECTO_DRIVE)
    else:
        raiz=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv").is_file()),None)
    if raiz is None: raise FileNotFoundError("No se encontró la raíz")
    return raiz

RAIZ=encontrar_raiz()
RUTA_DATOS=RAIZ/"proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv"
RUTA_SPLIT=RAIZ/"proyecto-integrador/04_regresion/salidas_s6/asignacion_split_viajes.csv"
RUTA_RF=RAIZ/"proyecto-integrador/05_clasificacion/salidas_s7/metricas_test_clasificacion_s7.csv"
SALIDAS=RAIZ/"proyecto-integrador/08_mlp_final/salidas_s10"
df=pd.read_csv(RUTA_DATOS,encoding="utf-8-sig"); split=pd.read_csv(RUTA_SPLIT,encoding="utf-8-sig")
df["particion"]=df["viaje_id"].map(split.set_index("viaje_id")["particion"])
df["apta_clasificacion_60min"]=df["apta_clasificacion_60min"].astype("boolean")
modelado=df.loc[df["apta_clasificacion_60min"].fillna(False)&df[TARGET].isin([0,1])].copy(); modelado[TARGET]=modelado[TARGET].astype(int)
print("TensorFlow",tf.__version__,"| observaciones",len(modelado))


TensorFlow 2.20.0 | observaciones 28557


## 2. Variables y particiones

La MLP utiliza las mismas variables base de S7–S9. Los predictores futuros permanecen excluidos.

In [2]:
NUMERICAS=["temp_c","humedad_pct","objetivo_c","tolerancia_c","desvio_respecto_umbral_c","temp_lag1_c","temp_lag2_c","cambio_temp_c","pendiente_c_por_min","temp_media_historica_3","temp_max_historica_3","minutos_desde_inicio","capacidad_kg_tratada","cantidad_solicitada_tratado","tiempo_entrega_prometido_hrs_tratado"]
CATEGORICAS=["categoria_logistica_tratada","tipo_camion_tratado","centro_distribucion_tratado"]
FEATURES=NUMERICAS+CATEGORICAS
PROHIBIDAS={TARGET,"max_desvio_termico_proximos_60min_c","n_lecturas_futuras_60min","apta_clasificacion_60min","apta_regresion_60min"}
assert not set(FEATURES)&PROHIBIDAS
ajuste=modelado.loc[modelado.particion.eq("AJUSTE")].copy(); validacion=modelado.loc[modelado.particion.eq("VALIDACION")].copy(); test=modelado.loc[modelado.particion.eq("TEST")].copy()
assert set(ajuste.viaje_id).isdisjoint(validacion.viaje_id) and set(ajuste.viaje_id).isdisjoint(test.viaje_id)
print("Ajuste/validación/test:",len(ajuste),len(validacion),len(test))


Ajuste/validación/test: 18946 4287 4274


## 3. Preprocesamiento y evaluación

El preprocesamiento aprende únicamente de AJUSTE. La búsqueda de umbral recorre 0,02 a 0,80 en VALIDACIÓN.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,average_precision_score,roc_auc_score,confusion_matrix

pre=ColumnTransformer([("num",Pipeline([("imputar",SimpleImputer(strategy="median",add_indicator=True)),("escalar",StandardScaler())]),NUMERICAS),("cat",Pipeline([("imputar",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),CATEGORICAS)],sparse_threshold=0)
X_aj=pre.fit_transform(ajuste[FEATURES]).astype("float32"); X_val=pre.transform(validacion[FEATURES]).astype("float32"); X_test=pre.transform(test[FEATURES]).astype("float32")
y_aj=ajuste[TARGET].to_numpy(dtype="float32"); y_val=validacion[TARGET].to_numpy(dtype="float32"); y_test=test[TARGET].to_numpy(dtype="float32")
assert np.isfinite(X_aj).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()

def metricas(real,prob,umbral=.5):
    pred=(np.asarray(prob)>=umbral).astype(int); tn,fp,fn,tp=confusion_matrix(real,pred,labels=[0,1]).ravel()
    return {"umbral":float(umbral),"accuracy":accuracy_score(real,pred),"precision":precision_score(real,pred,zero_division=0),"recall":recall_score(real,pred,zero_division=0),"f1":f1_score(real,pred,zero_division=0),"pr_auc":average_precision_score(real,prob),"roc_auc":roc_auc_score(real,prob),"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp),"alertas":int(pred.sum()),"costo_unidades":int(COSTO_FN*fn+COSTO_FP*fp)}

def tabla_umbrales(real,prob):
    filas=[]
    for u in np.linspace(.02,.80,157): filas.append(metricas(real,prob,float(u)))
    return pd.DataFrame(filas)

dummy=DummyClassifier(strategy="prior",random_state=SEMILLA); dummy.fit(X_aj,y_aj)
prob_dummy_val=dummy.predict_proba(X_val)[:,1]; met_dummy_val=metricas(y_val,prob_dummy_val,.5)
print("Baseline validación",met_dummy_val)


Baseline validación {'umbral': 0.5, 'accuracy': 0.9647772334966177, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'pr_auc': np.float64(0.03522276650338232), 'roc_auc': np.float64(0.5), 'TN': 4136, 'FP': 0, 'FN': 151, 'TP': 0, 'alertas': 0, 'costo_unidades': 1510}


## 4. MLP inicial y regularizada

La regularizada combina L2, Dropout y EarlyStopping con restauración de mejores pesos.

In [4]:
def crear_inicial():
    tf.keras.backend.clear_session(); tf.random.set_seed(SEMILLA)
    m=keras.Sequential([keras.Input(shape=(X_aj.shape[1],)),layers.Dense(32,activation="relu"),layers.Dense(16,activation="relu"),layers.Dense(1,activation="sigmoid")],name="mlp_inicial_s10")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=[keras.metrics.AUC(curve="PR",name="pr_auc"),keras.metrics.Recall(name="recall")]); return m

def crear_regularizada():
    tf.keras.backend.clear_session(); tf.random.set_seed(SEMILLA); reg=regularizers.l2(.001)
    m=keras.Sequential([keras.Input(shape=(X_aj.shape[1],)),layers.Dense(32,activation="relu",kernel_regularizer=reg),layers.Dropout(.20),layers.Dense(16,activation="relu",kernel_regularizer=reg),layers.Dense(1,activation="sigmoid")],name="mlp_regularizada_s10")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=[keras.metrics.AUC(curve="PR",name="pr_auc"),keras.metrics.Recall(name="recall")]); return m

inicial=crear_inicial(); hist_i=inicial.fit(X_aj,y_aj,validation_data=(X_val,y_val),epochs=60,batch_size=32,verbose=0)
early=keras.callbacks.EarlyStopping(monitor="val_loss",mode="min",min_delta=1e-3,patience=8,restore_best_weights=True)
regularizada=crear_regularizada(); hist_r=regularizada.fit(X_aj,y_aj,validation_data=(X_val,y_val),epochs=100,batch_size=32,callbacks=[early],verbose=0)
hi=pd.DataFrame(hist_i.history); hr=pd.DataFrame(hist_r.history)
print("Inicial épocas/mejor:",len(hi),int(hi.val_loss.idxmin()+1),float(hi.val_loss.min()))
print("Regularizada épocas/mejor:",len(hr),int(hr.val_loss.idxmin()+1),float(hr.val_loss.min()))



Inicial épocas/mejor: 60 7 0.11972629278898239
Regularizada épocas/mejor: 27 23 0.1206846535205841


## 5. Selección congelada

Se elige la MLP y el umbral de menor costo en VALIDACIÓN; en empate se prioriza recall.

In [5]:
prob_i=inicial.predict(X_val,verbose=0).ravel(); prob_r=regularizada.predict(X_val,verbose=0).ravel()
tu_i=tabla_umbrales(y_val,prob_i); tu_r=tabla_umbrales(y_val,prob_r)
mejor_i=tu_i.sort_values(["costo_unidades","recall"],ascending=[True,False]).iloc[0]
mejor_r=tu_r.sort_values(["costo_unidades","recall"],ascending=[True,False]).iloc[0]
comparacion_val=pd.DataFrame([{"modelo":"DummyClassifier",**met_dummy_val},{"modelo":"MLP inicial",**mejor_i.to_dict()},{"modelo":"MLP regularizada",**mejor_r.to_dict()}])
candidatos=comparacion_val.loc[comparacion_val.modelo.str.startswith("MLP")]
elegida=candidatos.sort_values(["costo_unidades","recall"],ascending=[True,False]).iloc[0]
NOMBRE_FINAL=str(elegida.modelo); UMBRAL_FINAL=float(elegida.umbral); MODELO_FINAL=inicial if NOMBRE_FINAL=="MLP inicial" else regularizada
print(comparacion_val.round(4).to_string(index=False)); print("CONGELADO:",NOMBRE_FINAL,"umbral",UMBRAL_FINAL)


          modelo  umbral  accuracy  precision  recall     f1  pr_auc  roc_auc     TN   FP    FN   TP  alertas  costo_unidades
 DummyClassifier    0.50    0.9648     0.0000  0.0000 0.0000  0.0352   0.5000 4136.0  0.0 151.0  0.0      0.0          1510.0
     MLP inicial    0.35    0.9715     0.7231  0.3113 0.4352  0.3544   0.7446 4118.0 18.0 104.0 47.0     65.0          1058.0
MLP regularizada    0.10    0.9708     0.6711  0.3377 0.4493  0.3698   0.7656 4111.0 25.0 100.0 51.0     76.0          1025.0
CONGELADO: MLP regularizada umbral 0.1


## 6. Apertura única de TEST

La configuración congelada se compara con Dummy y con Random Forest S7 sin volver a ajustar decisiones.

In [6]:
# Apertura única de TEST después de congelar modelo y umbral.
if globals().get("_TEST_ABIERTO",False): raise RuntimeError("TEST ya fue abierto en esta sesión")
_TEST_ABIERTO=True
prob_test=MODELO_FINAL.predict(X_test,verbose=0).ravel(); met_test=metricas(y_test,prob_test,UMBRAL_FINAL)
prob_dummy_test=dummy.predict_proba(X_test)[:,1]; met_dummy_test=metricas(y_test,prob_dummy_test,.5)
rf=pd.read_csv(RUTA_RF,encoding="utf-8-sig"); rf=rf.loc[~rf.modelo.eq("Dummy mayoritaria")].iloc[0]
comparacion_test=pd.DataFrame([{"modelo":"DummyClassifier",**met_dummy_test},{"modelo":NOMBRE_FINAL,**met_test},{"modelo":"Random Forest S7","umbral":rf.umbral,"accuracy":rf.accuracy,"precision":rf.precision,"recall":rf.recall,"f1":rf.f1,"pr_auc":rf.pr_auc,"roc_auc":rf.roc_auc,"TN":rf.TN,"FP":rf.FP,"FN":rf.FN,"TP":rf.TP,"alertas":rf.alertas,"costo_unidades":COSTO_FN*int(rf.FN)+COSTO_FP*int(rf.FP)}])
print(comparacion_test.round(4).to_string(index=False))


          modelo  umbral  accuracy  precision  recall     f1  pr_auc  roc_auc   TN  FP  FN  TP  alertas  costo_unidades
 DummyClassifier     0.5    0.9457     0.0000  0.0000 0.0000  0.0543   0.5000 4042   0 232   0        0            2320
MLP regularizada     0.1    0.9607     0.7133  0.4612 0.5602  0.5010   0.8503 3999  43 125 107      150            1293
Random Forest S7     0.7    0.9635     0.7836  0.4526 0.5738  0.5256   0.8437 4013  29 127 105      134            1299


## 7. Evidencias finales

In [7]:
SALIDAS.mkdir(parents=True,exist_ok=True)
comparacion_val.to_csv(SALIDAS/"comparacion_validacion_s10.csv",index=False,encoding="utf-8-sig"); comparacion_test.to_csv(SALIDAS/"comparacion_test_final_s10.csv",index=False,encoding="utf-8-sig")
tu_i.assign(modelo="MLP inicial").to_csv(SALIDAS/"umbrales_mlp_inicial_s10.csv",index=False,encoding="utf-8-sig"); tu_r.assign(modelo="MLP regularizada").to_csv(SALIDAS/"umbrales_mlp_regularizada_s10.csv",index=False,encoding="utf-8-sig")
hi.assign(epoca=np.arange(1,len(hi)+1)).to_csv(SALIDAS/"historial_mlp_inicial_s10.csv",index=False,encoding="utf-8-sig"); hr.assign(epoca=np.arange(1,len(hr)+1)).to_csv(SALIDAS/"historial_mlp_regularizada_s10.csv",index=False,encoding="utf-8-sig")
pred=(prob_test>=UMBRAL_FINAL).astype(int); salida=test[["fila_bronze","viaje_id","timestamp_bolivia",TARGET]].copy(); salida["probabilidad_mlp"]=prob_test; salida["umbral_congelado"]=UMBRAL_FINAL; salida["prediccion_mlp"]=pred; salida["tipo_resultado"]=np.select([salida[TARGET].eq(1)&salida.prediccion_mlp.eq(1),salida[TARGET].eq(0)&salida.prediccion_mlp.eq(0),salida[TARGET].eq(0)&salida.prediccion_mlp.eq(1)],["TP","TN","FP"],default="FN"); salida.to_csv(SALIDAS/"predicciones_test_mlp_s10.csv",index=False,encoding="utf-8-sig")

fig,axes=plt.subplots(1,2,figsize=(12,4)); axes[0].plot(hi.loss,label="Inicial train"); axes[0].plot(hi.val_loss,label="Inicial val"); axes[0].set(title="MLP inicial",xlabel="Época",ylabel="Pérdida"); axes[0].legend(); axes[1].plot(hr.loss,label="Regularizada train"); axes[1].plot(hr.val_loss,label="Regularizada val"); axes[1].set(title="MLP regularizada",xlabel="Época",ylabel="Pérdida"); axes[1].legend(); plt.tight_layout(); plt.savefig(SALIDAS/"curvas_regularizacion_s10.png",dpi=150,bbox_inches="tight"); plt.close()
cm=confusion_matrix(y_test,pred,labels=[0,1]); fig,ax=plt.subplots(figsize=(5,4)); ax.imshow(cm,cmap="Blues");
for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha="center",va="center",color="white" if v>cm.max()/2 else "black")
ax.set(xticks=[0,1],yticks=[0,1],xlabel="Predicho",ylabel="Real",title=f"TEST · {NOMBRE_FINAL} · umbral {UMBRAL_FINAL:.2f}"); plt.tight_layout(); plt.savefig(SALIDAS/"matriz_confusion_test_s10.png",dpi=150,bbox_inches="tight"); plt.close()
print("Salidas guardadas en",SALIDAS)


Salidas guardadas en c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\08_mlp_final\salidas_s10


## 8. Alcance

Los costos son didácticos y los datos sintéticos. La recomendación final debe distinguir desempeño experimental de autorización para operación.